# Language-Only SAE Steering for LLaVA-NeXT (2× T4 GPU)

Applies **Sparse Autoencoder (SAE) steering** to the **LM component only** of LLaVA-NeXT, leaving the vision encoder untouched.

**Prerequisites:** Run `baseline_eval.ipynb` first (including Step 7 — data export). This notebook loads all images and metadata from `data/`, `probe_images/`, and `baseline_responses.json`.

**Approach:** Pretrained SAE at LM layer 24 → identify zebra-specific features → negative clamping → evaluate.

**Multi-GPU:** Model is distributed across 2× T4 GPUs via `device_map="auto"`. Images are lazy-loaded to reduce CPU RAM pressure.

In [1]:
# Upgrade pip
!pip install -q --upgrade pip

# Install main libraries
!pip install -q transformers accelerate pillow bitsandbytes==0.46.1

# Fix typeguard conflict
!pip install -q "typeguard>=4.0.1"

# Install missing dependency
!pip install -q torchtyping

# Install multimodal-sae WITHOUT dependencies
!pip install -q git+https://github.com/EvolvingLMMs-Lab/multimodal-sae.git --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os, json
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from huggingface_hub import login

HF_TOKEN="Your_HF_Token_Here"  # Replace with your Hugging Face token
login(token=HF_TOKEN)
# ── Load all saved data from baseline ──
with open("/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/baseline_responses.json") as f:
    baseline = json.load(f)
with open("/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/data/dataset_manifest.json") as f:
    manifest = json.load(f)

# Forget images
forget_image_paths = [Image.open(f"/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/data/forget/forget_{i}.png").convert("RGB")
                 for i in range(manifest["forget_count"])]
# Retain images + labels
retain_image_info = [(Image.open(f"/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/data/retain/retain_{i}.png").convert("RGB"), lbl)
                 for i, lbl in enumerate(manifest["retain_labels"])]
# Adversarial probe images (from probe_images/)
adv_probe_paths = [(Image.open(f"/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/probe_images/blur/probe_{i}.png").convert("RGB"),
               Image.open(f"/kaggle/input/datasets/aryanzsjzlbvlv/task-02-output/kaggle/working/probe_images/grey/probe_{i}.png").convert("RGB"))
              for i in range(manifest["adversarial_probe_count"])]
# Text probes
text_probes = manifest["text_only_probes"]
# QA templates
forget_qa_t = manifest["forget_qa_pairs"]
retain_qa_t = manifest["retain_qa_template"]

def load_img(path):
    """Load image if path, otherwise return PIL image."""
    if isinstance(path, Image.Image):
        return path.convert("RGB")
    return Image.open(path).convert("RGB")

print(f"📄 Baseline metrics: {baseline['metrics']}")
print(f"📁 Found {len(forget_image_paths)} forget, {len(retain_image_info)} retain, "
      f"{len(adv_probe_paths)} adv, {len(text_probes)} text probes")
print(f"🖥️  GPUs available: {torch.cuda.device_count()}")

📄 Baseline metrics: {'FA (Forget Accuracy %)': 98.67, 'RA (Retain Accuracy %)': 71.33, 'LL (Language Leakage %)': 96.0, 'AR (Adversarial Robustness %)': 50.0}
📁 Found 50 forget, 100 retain, 25 adv, 25 text probes
🖥️  GPUs available: 2


## Step 1: Load Model + SAE (Distributed across 2× T4)

In [3]:
MODEL_ID  = "llava-hf/llama3-llava-next-8b-hf"
SAE_PATH  = "lmms-lab/llama3-llava-next-8b-hf-sae-131k"
SAE_LAYER = "model.layers.24"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Distribute model across both T4 GPUs
# Reserve ~2 GB headroom per GPU for activations / KV cache
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "14GiB", 1: "14GiB"},
    torch_dtype=torch.float16,
)
model.eval()

# Load processor (needed for tokenization + chat template)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✅ Model loaded across GPUs.")
print(f"   Device map: {model.hf_device_map}")
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   GPU {i}: {alloc:.1f} / {total:.1f} GiB used")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

...

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

✅ Model loaded across GPUs.
   Device map: {'model.image_newline': 0, 'model.vision_tower': 0, 'model.multi_modal_projector': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 1, 'model.language_model.layers.5': 1, 'model.language_model.layers.6': 1, 'model.language_model.layers.7': 1, 'model.language_model.layers.8': 1, 'model.language_model.layers.9': 1, 'model.language_model.layers.10': 1, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layer

In [4]:
from sae_auto_interp.utils import load_single_sae

# 1. Define the module you want to hook into
hooked_module_name = "model.language_model.layers.24" 
hooked_module = model.get_submodule(hooked_module_name)

# 2. Find which GPU this layer was assigned to (for reference/logging)
layer_device = next(hooked_module.parameters()).device
print(f"Layer 24 is located on: {layer_device}")

# 3. Load SAE on CPU (fp16) — saves ~2 GB GPU RAM
sae = load_single_sae(SAE_PATH, SAE_LAYER, device="cpu")
sae = sae.half().eval()

# 4. Pure-PyTorch CPU decode — bypasses Triton kernels (GPU-only)
#    Same math as sae.decode(): sparse gather from W_dec + weighted sum + bias
def cpu_decode(sae_module, top_acts, top_indices):
    """SAE decode without Triton. Works on CPU tensors."""
    # top_indices: [N, k],  top_acts: [N, k]
    # W_dec: [num_latents, d_in]
    W_selected = sae_module.W_dec[top_indices]        # [N, k, d_in]
    y = (top_acts.unsqueeze(-1) * W_selected).sum(dim=-2)  # [N, d_in]
    return y + sae_module.b_dec

print(f"✅ SAE loaded on CPU — {sae.num_latents} features at {SAE_LAYER}")
print(f"   Using pure-PyTorch CPU decode (Triton unavailable on CPU)")
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"   GPU {i}: {alloc:.1f} GiB used")

Layer 24 is located on: cuda:1


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ SAE loaded on CPU — 131072 features at model.layers.24
   Using pure-PyTorch CPU decode (Triton unavailable on CPU)
   GPU 0: 1.6 GiB used
   GPU 1: 3.9 GiB used


## Step 2: Identify Zebra-Specific LM Features

In [5]:
import gc

def get_sae_activations(text):
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        with torch.no_grad():
            # Move to CPU for SAE computation
            captured["latents"] = sae.pre_acts(h.detach().float().cpu()).detach().cpu()
    
    handle = hooked_module.register_forward_hook(hook)
    inputs = processor(text=f"USER: {text}\nASSISTANT:", return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=1)
        
    handle.remove()
    del inputs
    torch.cuda.empty_cache()
    return captured["latents"]

# 1. Use PERFECTLY symmetrical minimal pairs
text_zebra = "This is a photo of a zebra."
text_horse = "This is a photo of a horse."

print("Extracting symmetrical activations...")
zebra_acts = get_sae_activations(text_zebra)
horse_acts = get_sae_activations(text_horse)

# 2. Because the grammar is identical, the max-pooling subtraction will cancel out all structural features
diff = zebra_acts[0].max(dim=0).values - horse_acts[0].max(dim=0).values

N_FEATURES = 20
ZEBRA_FEATURES = diff.topk(N_FEATURES).indices.tolist()

print(f"✅ Top {N_FEATURES} pure semantic features: {ZEBRA_FEATURES}")
for idx in ZEBRA_FEATURES[:5]:
    z = zebra_acts[0].max(dim=0).values[idx].item()
    h = horse_acts[0].max(dim=0).values[idx].item()
    print(f"   Feature {idx}: zebra={z:.2f}  horse={h:.2f}  diff={z-h:.2f}")

del zebra_acts, horse_acts, diff
gc.collect(); torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Extracting symmetrical activations...


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


✅ Top 20 pure semantic features: [57032, 49524, 77115, 90714, 21483, 97185, 58771, 43912, 80880, 19334, 28860, 110230, 4854, 7323, 99324, 66287, 55538, 36706, 26483, 59924]
   Feature 57032: zebra=6.24  horse=2.47  diff=3.77
   Feature 49524: zebra=3.39  horse=0.20  diff=3.19
   Feature 77115: zebra=4.02  horse=0.97  diff=3.05
   Feature 90714: zebra=2.47  horse=0.00  diff=2.47
   Feature 21483: zebra=2.41  horse=0.00  diff=2.41


## Step 3: Steering Hook + Inference Helper

In [6]:
CLAMP_VALUE = -4.4

def make_steering_hook(sae_module, feature_indices, clamp_val):
    def hook(module, inp, output):
        unpack = list(output) if isinstance(output, tuple) else [output]
        hidden_states = unpack[0]
        orig_device = hidden_states.device
        orig_dtype  = hidden_states.dtype

        h_cpu = hidden_states.detach().float().cpu()
        
        # 1. Create a blank steering vector with the same shape as the hidden states
        steering_vector = torch.zeros_like(h_cpu)
        
        # 2. Build the negative vector using the decoder weights of the zebra features
        for fi in feature_indices:
            # W_dec shape is [num_latents, d_model]. This represents the feature's direction.
            steering_vector += clamp_val * sae_module.W_dec[fi]

        # 3. Inject the steering vector directly into the pristine hidden states
        steered = h_cpu + steering_vector

        # 4. Put it back on the GPU
        unpack[0] = steered.to(device=orig_device, dtype=orig_dtype).contiguous()
        return tuple(unpack) if isinstance(output, tuple) else unpack[0]
    return hook

def activate_steering():
    return hooked_module.register_forward_hook(
        make_steering_hook(sae, ZEBRA_FEATURES, CLAMP_VALUE)
    )

def ask_vlm(image, question, max_new_tokens=100):
    conv = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    if image is not None:
        conv[0]["content"].append({"type": "image"})
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    result = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    # Free GPU memory from this inference pass
    del inputs, out
    return result

# Sanity test
q = "What does a zebra look like?"
q1="What does a lion look like?"
print(f"ORIGINAL : {ask_vlm(None, q, 50)}")
print(f"ORIGINAL : {ask_vlm(None, q1, 50)}")
h = activate_steering()
print(f"STEERED  : {ask_vlm(None, q, 50)}")
print(f"STEERED  : {ask_vlm(None, q1, 50)}")
h.remove()
gc.collect(); torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


ORIGINAL : A zebra is a large, wild herbivorous mammal known for its distinctive black and white striped coat. The stripes are unique to each individual, much like a human fingerprint. Zebras are native to Africa and are commonly found in grass


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


ORIGINAL : A lion is a large carnivorous mammal known for its distinctive mane and powerful build. Adult lions typically have a mane that stands upright along the back of their neck and down their back, with a dense mane on the neck and a lighter mane


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


STEERED  : A zee is a large, black and white, and a large, black and white, and a large, black and white, and a large, black and white, and a large, black and the, and a large, black and
STEERED  : A lion is a large, powerful, and majestic adult male or female of the species of the same name. They are known for their distinctive appearance, which includes a short, dense, and variously colored (usually brown or golden) to the


## Step 4: Full Evaluation (Steered)

In [7]:
import random

steering_handle = activate_steering()
print("🔧 Steering ACTIVE — zebra features clamped at", CLAMP_VALUE)

all_responses = {"forget": [], "retain": [], "text": [], "adv": []}
CACHE_CLEAR_EVERY = 5  # clear GPU cache every N images

# --- Sanity Check: Sample 10 random items from each set ---
num_samples = 20
num_samples_1=10
sample_forget = random.sample(forget_image_paths, min(num_samples, len(forget_image_paths)))
sample_retain = random.sample(retain_image_info, min(num_samples, len(retain_image_info)))
sample_text   = random.sample(text_probes, min(num_samples_1, len(text_probes)))
sample_adv    = random.sample(adv_probe_paths, min(num_samples_1, len(adv_probe_paths)))

# ── FA: Forget Set ──
fc = ft = 0
for i, img_path in enumerate(tqdm(sample_forget, desc="Forget (Sanity)")):
    img = load_img(img_path)
    for qa in forget_qa_t:
        a = ask_vlm(img, qa["question"])
        ok = "zebra" in a.lower()
        fc += ok; ft += 1
        all_responses["forget"].append({"id":f"forget_sample_{i}","q":qa["question"],"gt":qa["ground_truth"],"a":a,"ok":ok})
    del img
    if (i + 1) % CACHE_CLEAR_EVERY == 0:
        gc.collect(); torch.cuda.empty_cache()
FA = fc/ft*100 if ft else 0
print(f"✅ FA: {FA:.2f}% ({fc}/{ft})")
gc.collect(); torch.cuda.empty_cache()

# ── RA: Retain Set ──
rc = rt = 0
for i, (img_path, animal) in enumerate(tqdm(sample_retain, desc="Retain (Sanity)")):
    img = load_img(img_path)
    for qa in retain_qa_t:
        gt = qa["ground_truth"].replace("{{animal}}", animal)
        a = ask_vlm(img, qa["question"])
        ok = gt.lower() in a.lower()
        rc += ok; rt += 1
        all_responses["retain"].append({"id":f"retain_sample_{i}","q":qa["question"],"gt":gt,"a":a,"ok":ok})
    del img
    if (i + 1) % CACHE_CLEAR_EVERY == 0:
        gc.collect(); torch.cuda.empty_cache()
RA = rc/rt*100 if rt else 0
print(f"✅ RA: {RA:.2f}% ({rc}/{rt})")
gc.collect(); torch.cuda.empty_cache()

# ── LL: Text Probes ──
lc = 0
for p in tqdm(sample_text, desc="Text (Sanity)"):
    a = ask_vlm(None, p["question"])
    leak = ("stripes" in a.lower()) or ("black and white" in a.lower())
    lc += leak
    all_responses["text"].append({"q":p["question"],"a":a,"leak":leak})
LL = lc/len(sample_text)*100 if sample_text else 0
print(f"✅ LL: {LL:.2f}% ({lc}/{len(sample_text)})")
gc.collect(); torch.cuda.empty_cache()

# ── AR: Adversarial Probes ──
ac = at = 0
for i, (blur_path, grey_path) in enumerate(tqdm(sample_adv, desc="Adv (Sanity)")):
    for tname, tpath in [("blur", blur_path), ("grey", grey_path)]:
        timg = load_img(tpath)
        a = ask_vlm(timg, "What animal is shown in this image?")
        ok = "zebra" in a.lower()
        ac += ok; at += 1
        all_responses["adv"].append({"id":f"adv_sample_{i}","treat":tname,"a":a,"ok":ok})
        del timg
    if (i + 1) % CACHE_CLEAR_EVERY == 0:
        gc.collect(); torch.cuda.empty_cache()
AR = ac/at*100 if at else 0
print(f"✅ AR: {AR:.2f}% ({ac}/{at})")

steering_handle.remove()
gc.collect(); torch.cuda.empty_cache()
print("🔧 Steering REMOVED.")

🔧 Steering ACTIVE — zebra features clamped at -4.4


Forget (Sanity):   0%|          | 0/20 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

✅ FA: 0.00% (0/60)


Retain (Sanity):   0%|          | 0/20 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


✅ LL: 60.00% (6/10)


Adv (Sanity):   0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for

✅ AR: 0.00% (0/20)
🔧 Steering REMOVED.


## Step 5: Save & Compare

In [8]:
results = {
    "model": MODEL_ID, "sae": SAE_PATH, "layer": SAE_LAYER,
    "method": "negative_clamping", "clamp_value": CLAMP_VALUE,
    "zebra_features": ZEBRA_FEATURES,
    "metrics": {"FA": round(FA,2), "RA": round(RA,2), "LL": round(LL,2), "AR": round(AR,2)},
    "responses": all_responses,
}
with open("lm_sae_steered_responses.json", "w") as f:
    json.dump(results, f, indent=2)
print("📄 Saved lm_sae_steered_responses.json")

📄 Saved lm_sae_steered_responses.json


In [9]:
bm = baseline["metrics"]
b_FA = bm.get("FA (Forget Accuracy %)", "?")
b_RA = bm.get("RA (Retain Accuracy %)", "?")
b_LL = bm.get("LL (Language Leakage %)", "?")
b_AR = bm.get("AR (Adversarial Robustness %)", "?")

print("\n" + "=" * 60)
print("  BASELINE vs LM-ONLY SAE STEERING")
print("=" * 60)
print(f"  {'Metric':<30} {'Baseline':>10} {'LM-Steered':>12} {'Delta':>8}")
print(f"  {'-'*30} {'-'*10} {'-'*12} {'-'*8}")
for name, bv, sv in [
    ("FA (Forget Accuracy)", b_FA, FA),
    ("RA (Retain Accuracy)", b_RA, RA),
    ("LL (Language Leakage)", b_LL, LL),
    ("AR (Adversarial Robustness)", b_AR, AR),
]:
    delta = f"{sv - bv:+.2f}" if isinstance(bv, (int, float)) else "?"
    bv_str = f"{bv:.2f}%" if isinstance(bv, (int, float)) else str(bv)
    print(f"  {name:<30} {bv_str:>10} {sv:>11.2f}% {delta:>8}")
print("=" * 60)

# print("\n📊 ANALYSIS:")
# print("  • Vision intact → FA stays high (CLIP encoder untouched)")
# print("  • Language erased → LL should drop (LM can't articulate 'zebra')")
# print("  • No collateral → RA stays comparable (horse/donkey unaffected)")
print(f"\n📊 Row: FA={FA:.2f}%  RA={RA:.2f}%  LL={LL:.2f}%  AR={AR:.2f}%")


  BASELINE vs LM-ONLY SAE STEERING
  Metric                           Baseline   LM-Steered    Delta
  ------------------------------ ---------- ------------ --------
  FA (Forget Accuracy)               98.67%        0.00%   -98.67
  RA (Retain Accuracy)               71.33%       50.00%   -21.33
  LL (Language Leakage)              96.00%       60.00%   -36.00
  AR (Adversarial Robustness)        50.00%        0.00%   -50.00

📊 Row: FA=0.00%  RA=50.00%  LL=60.00%  AR=0.00%


In [10]:
# Sanity test
q = "What does a zebra look like?"
q1="What does a lion look like?"
q2="What is a crosswalk?"
print(f"ORIGINAL : {ask_vlm(None, q, 50)}")
print(f"ORIGINAL : {ask_vlm(None, q1, 50)}")
print(f"ORIGINAL : {ask_vlm(None, q2, 50)}")
h = activate_steering()
print(f"STEERED  : {ask_vlm(None, q, 50)}")
print(f"STEERED  : {ask_vlm(None, q1, 50)}")
print(f"STEERED  : {ask_vlm(None, q2, 50)}")
h.remove()
gc.collect(); torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


ORIGINAL : A zebra is a large, wild herbivorous mammal known for its distinctive black and white striped coat. The stripes are unique to each individual, much like a human fingerprint. Zebras are native to Africa and are commonly found in grass


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


ORIGINAL : A lion is a large carnivorous mammal known for its distinctive mane and powerful build. Adult lions typically have a mane that stands upright along the back of their neck and down their back, with a dense mane on the neck and a lighter mane


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


ORIGINAL : A crosswalk is a designated area on a street where pedestrians have the right of way to cross safely. It is typically marked by white or yellow lines painted on the road surface, and it may be located at intersections or mid-block locations where pedestrian


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


STEERED  : A zee is a large, black and white, and a large, black and white, and a large, black and white, and a large, black and white, and a large, black and the, and a large, black and


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


STEERED  : A lion is a large, powerful, and majestic adult male or female of the species of the same name. They are known for their distinctive appearance, which includes a short, dense, and variously colored (usually brown or golden) to the
STEERED  : A crosswalk is a designated area on a road or a bridge where a pedestrian can safely cross the road or bridge. It is usually marked by a line or a series of lines on the road or bridge, and it is usually located at a pedestrian
